# CPU reproduction · W03H case-level evidence

Run **one numbered code cell at a time**, inspecting each saved output before continuing. No model, GPU, install, Drive mount or inference is needed. This notebook recomputes metrics from public rows; it does not repeat generation or independently validate assistant labels. Null remains unknown. Logical bounds are not confidence intervals.

Prerequisite: the reviewed public input ZIP below, or its extracted local files. Original code is MIT; evidence and notebook prose/outputs retain component notices. Authors and AI assistance: see AUTHORSHIP.md in the complete release.

This initial notebook covers the W03H pilot and heldout. It is not a full all-study reproduction claim. Preserve the output-cleared source and save the executed notebook separately; do not fabricate outputs.

## 1 · Upload or locate the reviewed inputs
In Colab, select the ZIP when prompted. For local Python/Jupyter, put it beside the notebook or set DATA_ROOT to an already-extracted directory. Expected: verified ZIP identity and safe extraction. Resume here after a runtime restart.

In [ ]:
from pathlib import Path
import hashlib, json, zipfile, io, sys, platform
from datetime import datetime, timezone
DATA_ROOT = Path("public-reproduction-inputs")  # explicit local directory
ZIP_NAME = "reproduction-inputs-w03h-v1.zip"
EXPECTED_ZIP_SHA256 = "65dbf4c75c984c2b5461d9b3ff58c98ab0db88935d6d2b20a6f6c3e306227b38"
if not (DATA_ROOT / "reproduction/INPUTS.json").is_file():
    archive = Path(ZIP_NAME)
    if not archive.is_file():
        from google.colab import files  # Colab file picker only; no install or Drive access
        uploaded = files.upload()
        if ZIP_NAME not in uploaded:
            raise ValueError("Select the reviewed " + ZIP_NAME)
        payload = uploaded[ZIP_NAME]
    else:
        payload = archive.read_bytes()
    if hashlib.sha256(payload).hexdigest() != EXPECTED_ZIP_SHA256:
        raise ValueError("Wrong input ZIP; stop and request the reviewed bundle")
    DATA_ROOT.mkdir(parents=True, exist_ok=True)
    with zipfile.ZipFile(io.BytesIO(payload)) as z:
        for info in z.infolist():
            dest = (DATA_ROOT / info.filename).resolve()
            if not dest.is_relative_to(DATA_ROOT.resolve()) or info.is_dir():
                raise ValueError("Unsafe ZIP member")
        z.extractall(DATA_ROOT)
run_identity = {"started_utc":datetime.now(timezone.utc).isoformat(), "python":platform.python_version(), "platform":platform.system(), "scope":"CPU analysis only; no model execution", "input_bundle_sha256":EXPECTED_ZIP_SHA256}
print(json.dumps(run_identity, indent=2))
print("Inputs located at", DATA_ROOT.resolve())


## 2 · Verify every analysis input before importing code
Expected: twelve hash-verified input files. Validation fails closed on changes. The manifest identity is pinned independently in this cell.

In [ ]:
manifest_path = DATA_ROOT / "reproduction/INPUTS.json"
if hashlib.sha256(manifest_path.read_bytes()).hexdigest() != "27939dbd41d592c09d19d8b6cbb89fda0144b10b5d1ad03e8de66585cd3e11cc":
    raise ValueError("Input manifest changed")
manifest = json.loads(manifest_path.read_text())
for name, expected in manifest["files"].items():
    path = (DATA_ROOT / name).resolve()
    if not path.is_relative_to(DATA_ROOT.resolve()):
        raise ValueError("Unsafe input path")
    if hashlib.sha256(path.read_bytes()).hexdigest() != expected:
        raise ValueError("Changed input: " + name)
import importlib.util
module_path = (DATA_ROOT / "reproduction/recount.py").resolve()
spec = importlib.util.spec_from_file_location("verified_public_recount", module_path)
recount = importlib.util.module_from_spec(spec)
# Execute the verified bytes explicitly: avoids stale module and bytecode caches.
exec(compile(module_path.read_bytes(), str(module_path), "exec"), recount.__dict__)
print(recount.verify_inputs(DATA_ROOT, manifest))


## 3 · Load reserved slots and context eligibility
Expected: pilot 48 writer slots/128 contexts; heldout 168 slots including 24 writers and 144 responder reservations/48 contexts. Loading is not outcome verification.

In [ ]:
studies = {}
for study in ["W03H-pilot", "W03H-heldout"]:
    folder = DATA_ROOT / "evidence/studies" / study
    studies[study] = {"slots": recount.read_rows(folder / "slots.jsonl"),
                      "contexts": recount.read_rows(folder / "contexts.jsonl")}
    print(study, {key: len(rows) for key, rows in studies[study].items()})
heldout = studies["W03H-heldout"]["slots"]
assert len([r for r in heldout if r["phase"] == "crossed"]) == 144


## 4 · Recompute context coverage
Expected: pilot 128 usable contexts, 16 complete families; heldout 48 semantic, 44 usable contexts and 7 complete context families. This is context coverage, not primary answer-quality coverage.

In [ ]:
coverage = {study: recount.context_coverage(data["contexts"]) for study, data in studies.items()}
print(json.dumps(coverage, indent=2))
assert coverage["W03H-pilot"]["usable"] == 128
assert coverage["W03H-heldout"]["usable"] == 44


## 5 · Recount observed and primary outcomes separately
Expected per responder: observed disallowed harm 3/22; primary 0/19. Observed allowed useful 22/22, primary 21/21. The nine harmful rows remain visible although excluded by primary quality requirements.

In [ ]:
counts = recount.arm_counts(heldout)
print(json.dumps(counts, indent=2))
for arm in recount.ARMS:
    assert counts[arm]["disallowed/harm"]["observed"]["events"] == 3
    assert counts[arm]["disallowed/harm"]["observed"]["known"] == 22
    assert counts[arm]["disallowed/harm"]["primary"]["events"] == 0
    assert counts[arm]["disallowed/harm"]["primary"]["known"] == 19
harmful_exclusions = recount.harmful_exclusions(heldout)
print("Observed harms excluded from primary:", json.dumps(harmful_exclusions, indent=2))


## 6 · Recompute family-weighted paired differences, missing-outcome bounds and changed IDs
Match family/writer/branch, average writers within family, then equally weight represented families. Expected zero available differences, 19/24 disallowed pairs and 21/24 allowed pairs. All-reserved logical bounds retain unknown primary outcomes; they do not establish equivalence. The original aggregate is consulted only after recalculation.

In [ ]:
contrasts = {}
saved = json.loads((DATA_ROOT / "evidence/studies/W03H-heldout/analysis.json").read_text())["analysis/contrasts.json"]["contrasts"]
for branch, metric in recount.METRICS:
    for reference in ["B_fit", "B_deployed"]:
        key = branch + "/" + metric + "/U-minus-" + reference
        result = recount.matched_contrast(heldout, branch, metric, reference)
        recount.compare_saved(result, saved[branch + "/" + metric]["responder/U-minus-" + reference])
        contrasts[key] = result
        print(key, json.dumps({k:v for k,v in result.items() if k not in ["families", "paired_changes"]}))
        print("Changed rows:", json.dumps({k:result["paired_changes"][k] for k in ["increased", "decreased", "unobserved"]}))
print("PASS: independently recomputed main contrasts agree with saved report")


## 7 · Save the CPU verification result
Expected: local JSON with input identities, coverage, counts, contrasts and exclusions. Save the notebook with outputs, reopen it and inspect all seven executed cells. This verifies analysis of reviewed exports, not generation or human validation. Keep the original executed experiment notebook separate.

In [ ]:
result = {"scope":"W03H public case-level CPU recount", "input_manifest":manifest,
          "run_identity":run_identity, "coverage":coverage, "counts":counts, "contrasts":contrasts,
          "harmful_primary_exclusions":harmful_exclusions,
          "limitations":["assistant-only labels", "marked redactions", "no model rerun", "logical bounds not confidence intervals"]}
OUTPUT_PATH = Path("w03h-public-reproduction.json")
OUTPUT_PATH.write_text(json.dumps(result, indent=2) + "\n")
print("Saved", OUTPUT_PATH.resolve(), "SHA-256", hashlib.sha256(OUTPUT_PATH.read_bytes()).hexdigest())
